In [13]:
import os
import random
import shutil
import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

In [2]:
import warnings 
warnings.filterwarnings('ignore')

In [3]:
# Set seed values for random number generator and file system

seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

In [4]:
# Define constants
SIZE = 150
img_width, img_height = SIZE, SIZE
batch_size = 32
TRAIN_DIR = "Model Data/Train"
VAL_DIR = "Model Data/Validation"
TEST_DIR = "Model Data/Test"

In [5]:
# Create an ImageDataGenerator for the training data
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

# Create an ImageDataGenerator for the validation data
validation_datagen = ImageDataGenerator(rescale=1./255)
validation_generator = validation_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

# Create an ImageDataGenerator for the test data
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

Found 4200 images belonging to 8 classes.
Found 904 images belonging to 8 classes.
Found 896 images belonging to 8 classes.


In [6]:
num_classes = 8

model = Sequential()
model.add(Conv2D(32, (3, 3), activation="relu", input_shape=(SIZE, SIZE, 3)))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2)))  
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3),activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2)))  
model.add(Dropout(0.25))

model.add(Conv2D(128, (3, 3),activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2)))  
model.add(Dropout(0.25))
\
model.add(Conv2D(128, (3, 3),activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2)))  
model.add(Dropout(0.25))

model.add(Conv2D(512, (3, 3),activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2)))  
model.add(Dropout(0.25))
model.add(Flatten())

model.add(Dense(64))
model.add(Dense(32))
model.add(Dense(16))
model.add(Dense(8, activation='softmax'))


In [7]:
mc = ModelCheckpoint(filepath="./best_model.h5",
                     monitor="accuracy",
                     verbose=1,
                     save_best_only=True
                    )

es = EarlyStopping(monitor="accuracy",
                    verbose=1,
                    min_delta=0.001,
                    patience=10
                  )

cb = [mc,es]

In [8]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 148, 148, 32)      896       
                                                                 
 batch_normalization (BatchN  (None, 148, 148, 32)     128       
 ormalization)                                                   
                                                                 
 max_pooling2d (MaxPooling2D  (None, 74, 74, 32)       0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 74, 74, 32)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 72, 72, 64)        18496     
                                                                 
 batch_normalization_1 (Batc  (None, 72, 72, 64)       2

In [9]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size,
    epochs=500,
    shuffle=True,
    callbacks=cb
)


Epoch 1/500
131/131 [==============================] - ETA: 0s - loss: 3.0906 - accuracy: 0.2435
Epoch 1: accuracy improved from -inf to 0.24352, saving model to .\best_model.h5
131/131 [==============================] - 59s 424ms/step - loss: 3.0906 - accuracy: 0.2435 - val_loss: 2.1924 - val_accuracy: 0.1272
Epoch 2/500
131/131 [==============================] - ETA: 0s - loss: 2.4208 - accuracy: 0.2781
Epoch 2: accuracy improved from 0.24352 to 0.27807, saving model to .\best_model.h5
131/131 [==============================] - 43s 326ms/step - loss: 2.4208 - accuracy: 0.2781 - val_loss: 2.2736 - val_accuracy: 0.1239
Epoch 3/500
131/131 [==============================] - ETA: 0s - loss: 2.1268 - accuracy: 0.3040
Epoch 3: accuracy improved from 0.27807 to 0.30398, saving model to .\best_model.h5
131/131 [==============================] - 43s 326ms/step - loss: 2.1268 - accuracy: 0.3040 - val_loss: 2.2979 - val_accuracy: 0.1261
Epoch 4/500
131/131 [==============================] - ETA

In [10]:
# Evaluate the model on the test data
test_loss, test_acc = model.evaluate(test_generator)
print('Test accuracy:', test_acc)
print('Test loss:', test_loss)

28/28 [==============================] - 6s 230ms/step - loss: 1.7620 - accuracy: 0.5737
Test accuracy: 0.5736607313156128
Test loss: 1.7620042562484741


In [15]:
# Load the saved model
model = load_model('best_model.h5')

In [16]:
# Plot the training and validation loss over epochs
plt.plot(model.history.history['loss'])
plt.plot(model.history.history['val_loss'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()


AttributeError: 'NoneType' object has no attribute 'history'

In [17]:
# Plot the training and validation accuracy over epochs
plt.plot(model.history.history['accuracy'])
plt.plot(model.history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

AttributeError: 'NoneType' object has no attribute 'history'